In [1]:
!pip install git+https://github.com/m-bain/whisperX.git

  Cloning https://github.com/m-bain/whisperX.git to /tmp/pip-req-build-appieqh4
  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperX.git /tmp/pip-req-build-appieqh4
  Resolved https://github.com/m-bain/whisperX.git to commit 5f2f9d4320dd93a7d12f5ba2495eef7e0a5af963
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [30]:
import google.colab as colab
import whisperx
from whisperx.diarize import DiarizationPipeline
import os
from google.colab import userdata

# Fetch token from Colab Secrets
hf_token = userdata.get('HF_TOKEN_1')

device = "cuda"

batch_size = 16
compute_type = "float16" # change to "int8" if running on a CPU runtime

model = whisperx.load_model("large-v3", device, compute_type=compute_type)

# Use the 'token' argument to pass your credentials directly
diarize_model = DiarizationPipeline(
    model_name="pyannote/speaker-diarization-3.1",
    token=hf_token,
    device=device
)


2026-06-17 02:50:15 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-06-17 02:50:15 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


2026-06-17 02:50:15 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-3.1


In [40]:
# 1. Transcribe with WhisperX
audio_file = "/content/Chapter 12.mp3"
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

2026-06-17 03:12:22 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio


In [41]:
# 2. Align whisper output to get word-level exact timestamps
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)



In [42]:
# 3. Diarize (Assign Speaker Labels)

diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)

In [43]:

# 4. Format and save as a structured text file
output_path = audio_file.split('.')[0] + ".txt"
with open(output_path, "w", encoding="utf-8") as f:
    current_speaker = None
    paragraph = []

    for segment in result["segments"]:
        speaker = segment.get("speaker", "UNKNOWN_SPEAKER")
        start_time = segment["start"]
        end_time = segment["end"]
        text = segment["text"].strip()

        # Convert seconds to HH:MM:SS format
        start_str = f"{int(start_time//3600):02d}:{int((start_time%3600)//60):02d}:{int(start_time%60):02d}"

        # Split paragraph if speaker changes
        if speaker != current_speaker:
            if paragraph:
                f.write(" ".join(paragraph) + "\n\n")
                paragraph = []
            f.write(f"[{start_str}] {speaker}:\n")
            current_speaker = speaker

        paragraph.append(text)

    # Write the final segment
    if paragraph:
        f.write(" ".join(paragraph) + "\n")

print(f"Structured file saved to {output_path}")

Structured file saved to /content/Chapter 12.txt
